# Deep Research Agent V2 — 高级模块化多轮检索

> BrowseComp-Plus + Qwen3-8B
>
> 相比 V1（简单 ReAct），V2 新增：Question Analyzer / Query Rewriter / Relevance Filter / Context Compressor / Verifier

## 0. 环境设置

In [ ]:
!python --version
!pip install -r agent/requirements.txt -q

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# ── vLLM 服务配置 ──────────────────────────────────
VLLM_BASE_URL = 'http://127.0.0.1:8000/v1'
MODEL_NAME = 'qwen_auto'
API_KEY = 'dummy'

# ── 路径 ──────────────────────────────────────────
CORPUS_PATH = 'browsecomp-plus-corpus'
BM25_INDEX_PATH = 'indexes/browsecomp_plus_bm25.sqlite'
HARD50_PATH = 'browsecomp_plus_hard50.jsonl'
OUTPUT_DIR = 'runs'

## 1. BM25 索引（仅首次）

In [ ]:
import os
if not os.path.exists(BM25_INDEX_PATH):
    !python -m agent.build_bm25_index --corpus-path {CORPUS_PATH} --index-path {BM25_INDEX_PATH} --overwrite
else:
    print(f'Index exists: {BM25_INDEX_PATH}')

## 2. 初始化

选择 Agent 版本：
- **V1**（简单 ReAct）— 适合快速验证 tool calling 链路
- **V2**（完整模块）— Question Analyzer + Query Rewriter + Relevance Filter + Context Compressor + Verifier

In [ ]:
AGENT_VERSION = 'V1'  # 切换: 'V1' 或 'V2'（V2 已修复，推荐先用 V1 跑通基线）

from agent.vllm_client import VLLMClient
from agent.tools import build_searcher
from agent.dataset_utils import load_jsonl

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
print('search_type:', searcher.search_type)

if AGENT_VERSION == 'V1':
    from agent.deep_research_agent import DeepResearchAgent
    agent = DeepResearchAgent(
        client=client, searcher=searcher, model_name=MODEL_NAME,
        max_rounds=8, max_tokens=2048, top_k=10, temperature=0.0,
    )
    print('Agent: V1 (Simple ReAct — 推荐先跑这个)')
else:
    from agent.deep_research_agent_v2 import DeepResearchAgentV2
    agent = DeepResearchAgentV2(
        client=client, searcher=searcher, model_name=MODEL_NAME,
        max_rounds=10, max_tokens=2048, top_k=15, temperature=0.0,
        use_analyzer=True,
        use_rewriter=True,
        use_filter=True,
        use_compressor=True,
        use_verifier=True,
        compress_after_round=5,
    )
    print('Agent: V2 (Analyzer + Rewriter + Filter + Compressor + Verifier)')

## 3. 单条测试

In [ ]:
rows = load_jsonl(HARD50_PATH, limit=50)
demo = rows[0]

print(f'query_id: {demo["query_id"]}')
print(f'query: {demo["query"][:200]}...')
print(f'gold: {demo["answer"]}')
print('---')

result = agent.solve(question=demo['query'], query_id=demo['query_id'])

print(f'\nPredicted: {result["predicted_answer"]}')
print(f'Status: {result["status"]}')
print(f'Rounds: {result["rounds_used"]}')
print(f'Tool calls: {result["num_tool_calls"]}')
if 'num_searches' in result:
    print(f'Searches: {result["num_searches"]}, Docs examined: {result["num_docs_examined"]}')
if 'verification' in result and result['verification']:
    print(f'Verification: {result["verification"]["verdict"]} — {result["verification"]["reasoning"]}')

## 3.5 轨迹调试 — 查看单题完整推理过程

调试时用，打印每个 tool call 的传入参数和返回结果。修改 `TARGET_QID` 可以切换题目。

## 4. 批量运行

In [ ]:
# ── 轨迹调试：查看 q442 完整推理过程 ──

def print_trajectory(result):
    qid = result.get('query_id', '?')
    ans = result.get('predicted_answer', '?')[:80]
    status = result.get('status', '?')
    rounds = result.get('rounds_used', '?')
    tc = result.get('num_tool_calls', '?')
    print(f'Query: {qid}  |  Status: {status}  |  Rounds: {rounds}  |  Tool calls: {tc}')
    print(f'Final answer: {ans}')
    print('=' * 70)

    msgs = result.get('messages', [])
    round_num = 0
    for i, msg in enumerate(msgs):
        role = msg['role']
        content = (msg.get('content') or '').strip()
        tool_calls = msg.get('tool_calls') or []

        if role == 'system':
            print(f'\n[SYSTEM] ({len(content)} chars)')
        elif role == 'user':
            # Skip full text, show preview
            preview = content[:200].replace('\n', ' | ')
            print(f'\n[USER] {preview}...' if len(content) > 200 else f'\n[USER] {preview}')
        elif role == 'assistant' and tool_calls:
            round_num += 1
            for tc in tool_calls:
                fn = tc.get('function', {})
                name = fn.get('name', '')
                args = fn.get('arguments', '{}')[:150]
                print(f'\n── Round {round_num} ──')
                print(f'[TOOL CALL] {name}({args})')
        elif role == 'assistant' and content:
            print(f'\n[ASSISTANT] {content[:300]}')
        elif role == 'tool':
            preview = content[:120].replace('\n', ' | ')
            print(f'  [TOOL RESULT] {preview}...' if len(content) > 120 else f'  [TOOL RESULT] {preview}')

# ── 跑 q442 ──
demo_q = [row for row in rows if row['query_id'] == '442']
if demo_q:
    print('>>> Running q442 with trajectory...\n')
    result = agent.solve(question=demo_q[0]['query'], query_id='442')
    print_trajectory(result)
else:
    print('q442 not found in dataset')

In [ ]:
import json, time
from datetime import datetime

timestamp = datetime.now().strftime('%m%d_%H%M')
output_path = str(Path(OUTPUT_DIR) / f'submission_{timestamp}.jsonl')
eval_output_path = str(Path(OUTPUT_DIR) / f'eval_results_{timestamp}.jsonl')
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f'Output: {output_path}')

records = []
t_start = time.time()

for i, row in enumerate(rows):
    print(f'[{i+1}/{len(rows)}] q={row["query_id"]}...', end=' ', flush=True)
    t0 = time.time()
    result = agent.solve(question=row['query'], query_id=row['query_id'])
    elapsed = time.time() - t0
    ans = result['predicted_answer'][:60]
    print(f'r={result["rounds_used"]} tc={result["num_tool_calls"]} '
          f'ans={ans}... ({elapsed:.1f}s)')
    records.append(result)

total_time = time.time() - t_start
with open(output_path, 'w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'\nDone. {len(records)} results → {output_path}')
print(f'Total: {total_time:.1f}s ({total_time/len(records):.1f}s/q)')

## 5. 自动评估

In [ ]:
from agent.eval import run_evaluation

summary, details = run_evaluation(
    submission_path=output_path,
    dataset_path=HARD50_PATH,
    model_name=MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key=API_KEY,
    output_path=eval_output_path,
    verbose=True,
)

In [ ]:
print(f"{'='*50}")
print(f"评估结果 (Agent {AGENT_VERSION})")
print(f"{'='*50}")
print(f"总题数:     {summary['total_queries']}")
print(f"正确:       {summary['correct']}")
print(f"错误:       {summary['incorrect']}")
print(f"准确率:     {summary['accuracy']:.2%}")
print(f"平均工具调用: {summary['avg_tool_calls_per_query']}")
print(f"平均检索文档: {summary['avg_retrieved_docs_per_query']}")
print()

# 错误归因
error_cases = [d for d in details if d['eval_judgment'] == 'INCORRECT']
patterns = {'insufficient_search': 0, 'wrong_query': 0, 'hallucination': 0, 'context_overload': 0}
for c in error_cases:
    tc = c['trajectory_stats']['num_tool_calls']
    if tc <= 1: patterns['insufficient_search'] += 1
    elif 'insufficient' in c['predicted_answer'].lower(): patterns['wrong_query'] += 1
    elif tc >= 8: patterns['context_overload'] += 1
    else: patterns['hallucination'] += 1

print("错误模式:")
for k, v in patterns.items():
    print(f"  {k}: {v} ({v/len(error_cases)*100:.0f}%)" if error_cases else f"  {k}: 0")